In [36]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 38, Finished, Available, Finished, False)

# **Working with Matches Table**

In [37]:
bronze_matches = spark.read.table("bronze_matches")

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 39, Finished, Available, Finished, False)

In [38]:
display(bronze_matches)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 441f2ecd-fe40-480f-b8f2-06e1929298cc)

In [39]:
bronze_matches.printSchema()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 41, Finished, Available, Finished, False)

root
 |-- eliminator: string (nullable = true)
 |-- team1: string (nullable = true)
 |-- neutralvenue: boolean (nullable = true)
 |-- balls_per_over: integer (nullable = true)
 |-- umpire2: string (nullable = true)
 |-- umpire1: string (nullable = true)
 |-- outcome: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- date1: string (nullable = true)
 |-- date2: string (nullable = true)
 |-- method: string (nullable = true)
 |-- date: date (nullable = true)
 |-- team2: string (nullable = true)
 |-- player_of_match: string (nullable = true)
 |-- winner_wickets: integer (nullable = true)
 |-- winner_runs: integer (nullable = true)
 |-- reserve_umpire: string (nullable = true)
 |-- season: string (nullable = true)
 |-- city: string (nullable = true)
 |-- winner: string (nullable = true)
 |-- match_number: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- match_referee: string (nullable = true)
 |-- tv_umpire: string (nu

In [40]:
bronze_matches.select([
    count(when (col(c).isNull(), c)).alias(c)
    for c in bronze_matches.columns
]).show()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 42, Finished, Available, Finished, False)

+----------+-----+------------+--------------+-------+-------+-------+-----+-----+-----+------+----+-----+---------------+--------------+-----------+--------------+------+----+------+------------+-----+------+-------------+---------+-----------+-------------+-------+-------------------+----------------+
|eliminator|team1|neutralvenue|balls_per_over|umpire2|umpire1|outcome|venue|date1|date2|method|date|team2|player_of_match|winner_wickets|winner_runs|reserve_umpire|season|city|winner|match_number|event|gender|match_referee|tv_umpire|toss_winner|toss_decision|matchId|ingestion_timestamp|source_file_name|
+----------+-----+------------+--------------+-------+-------+-------+-----+-----+-----+------+----+-----+---------------+--------------+-----------+--------------+------+----+------+------------+-----+------+-------------+---------+-----------+-------------+-------+-------------------+----------------+
|      1081|    0|        1018|             0|      0|      0|   1076|    0| 1093| 10

In [41]:
bronze_matches.select("team1").distinct().show(100, False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 43, Finished, Available, Finished, False)

+---------------------------+
|team1                      |
+---------------------------+
|Sunrisers Hyderabad        |
|Lucknow Super Giants       |
|Chennai Super Kings        |
|Gujarat Titans             |
|Royal Challengers Bengaluru|
|Rising Pune Supergiant     |
|Deccan Chargers            |
|Kochi Tuskers Kerala       |
|Rajasthan Royals           |
|Gujarat Lions              |
|Royal Challengers Bangalore|
|Kolkata Knight Riders      |
|Rising Pune Supergiants    |
|Kings XI Punjab            |
|Punjab Kings               |
|Pune Warriors              |
|Delhi Daredevils           |
|Delhi Capitals             |
|Mumbai Indians             |
+---------------------------+



In [42]:
bronze_matches.select("team2").distinct().show(100, False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 44, Finished, Available, Finished, False)

+---------------------------+
|team2                      |
+---------------------------+
|Sunrisers Hyderabad        |
|Lucknow Super Giants       |
|Chennai Super Kings        |
|Gujarat Titans             |
|Royal Challengers Bengaluru|
|Rising Pune Supergiant     |
|Deccan Chargers            |
|Kochi Tuskers Kerala       |
|Rajasthan Royals           |
|Gujarat Lions              |
|Royal Challengers Bangalore|
|Kolkata Knight Riders      |
|Rising Pune Supergiants    |
|Kings XI Punjab            |
|Punjab Kings               |
|Pune Warriors              |
|Delhi Daredevils           |
|Delhi Capitals             |
|Mumbai Indians             |
+---------------------------+



### Standardizing Team Names

In [43]:
# Create Mapping Dictionary
team_name_mapping = {
    "Delhi Daredevils" : "Delhi Capitals",
    "Rising Pune Supergiant" : "Rising Pune Supergiants",
    "Kings XI Punjab" : "Punjab Kings"
}

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 45, Finished, Available, Finished, False)

In [44]:
for old_name, new_name in team_name_mapping.items():

    bronze_matches=bronze_matches.replace(
        old_name,
        new_name,
        subset=["team1", "team2", "winner", "toss_winner"]
    )

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 46, Finished, Available, Finished, False)

In [45]:
columns = [
    "eliminator",
    "date1",
    "date2",
    "neutralvenue",
    "balls_per_over",
    "outcome",
    "method"
]

for col_name in columns:
    print(f"\nDistinct values in {col_name}:\n")
    
    bronze_matches.select(col_name)\
                  .distinct()\
                  .show(100, False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 47, Finished, Available, Finished, False)


Distinct values in eliminator:

+---------------------------+
|eliminator                 |
+---------------------------+
|Sunrisers Hyderabad        |
|Rajasthan Royals           |
|Royal Challengers Bangalore|
|Kolkata Knight Riders      |
|Kings XI Punjab            |
|Delhi Capitals             |
|Mumbai Indians             |
|NULL                       |
+---------------------------+


Distinct values in date1:

+----------+
|date1     |
+----------+
|2014/05/27|
|2017/05/17|
|NULL      |
+----------+


Distinct values in date2:

+----------+
|date2     |
+----------+
|2017/05/18|
|2014/05/28|
|NULL      |
+----------+


Distinct values in neutralvenue:

+------------+
|neutralvenue|
+------------+
|true        |
|NULL        |
+------------+


Distinct values in balls_per_over:

+--------------+
|balls_per_over|
+--------------+
|6             |
+--------------+


Distinct values in outcome:

+---------+
|outcome  |
+---------+
|tie      |
|no result|
|NULL     |
+---------+


D

In [46]:
# cleaned_matches = (
#     bronze_matches

#     # Standardise date format to proper DateType for time-series analysis in Gold
#     .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

#     # Fix 'eliminator': NULL means it was a league stage match, not a knockout
#     .withColumn("eliminator", when(col("eliminator").isNull(), lit("No")).otherwise(col("eliminator")))

#     # Fix 'outcome': NULL means match had a normal result (win by runs/wickets)
#     .withColumn("outcome", when(col("outcome").isNull(), lit("Normal")).otherwise(col("outcome")))

#     # Fix 'method': NULL means no DLS applied
#     .withColumn("method", when(col("method").isNull(), lit("Normal")).otherwise(col("method")))

#     # Standardise team names — trim whitespace and uppercase to avoid
#     # "Mumbai Indians" vs "mumbai indians" mismatches when joining with deliveries
#     .withColumn("team1", trim(col("team1")))
#     .withColumn("team2", trim(col("team2")))
#     .withColumn("winner", trim(col("winner")))
#     .withColumn("toss_winner", trim(col("toss_winner")))

#     # Drop columns with no analytical value
#     .drop("neutralvenue", "balls_per_over", "date1", "date2", "event")
# )

# display(cleaned_matches)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 48, Finished, Available, Finished, False)

In [47]:
# cleaned_matches.select("outcome", "winner_runs", "winner_wickets") \
#     .filter(col("winner_runs").isNull() & col("winner_wickets").isNull()) \
#     .select("outcome") \
#     .distinct() \
#     .show(truncate=False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 49, Finished, Available, Finished, False)

In [48]:
# cleaned_matches = cleaned_matches \
#     .withColumn("winner_runs", when(col("winner_runs").isNull(), lit(0)).otherwise(col("winner_runs"))) \
#     .withColumn("winner_wickets", when(col("winner_wickets").isNull(), lit(0)).otherwise(col("winner_wickets"))) \
#     .withColumn("result_type",
#         when(col("outcome") == "tie", lit("Tie"))
#         .when(col("outcome") == "no result", lit("No Result"))
#         .when(col("winner_wickets") > 0, lit("Won by Wickets"))
#         .when(col("winner_runs") > 0, lit("Won by Runs"))
#         .otherwise(lit("Unknown"))
#     )

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 50, Finished, Available, Finished, False)

In [49]:
bronze_matches.filter(col("city").isNull()).select("matchId", "venue", "city").show()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 51, Finished, Available, Finished, False)

+-------+--------------------+----+
|matchId|               venue|city|
+-------+--------------------+----+
| 729281|Sharjah Cricket S...|NULL|
| 729287|Dubai Internation...|NULL|
| 729289|Dubai Internation...|NULL|
| 729291|Sharjah Cricket S...|NULL|
| 729295|Sharjah Cricket S...|NULL|
| 729297|Dubai Internation...|NULL|
| 729299|Sharjah Cricket S...|NULL|
| 729303|Dubai Internation...|NULL|
| 729301|Dubai Internation...|NULL|
| 729309|Sharjah Cricket S...|NULL|
| 729311|Sharjah Cricket S...|NULL|
| 729313|Dubai Internation...|NULL|
| 729317|Dubai Internation...|NULL|
|1216493|Dubai Internation...|NULL|
|1216534|Dubai Internation...|NULL|
|1216496|Sharjah Cricket S...|NULL|
|1216510|Dubai Internation...|NULL|
|1216539|Dubai Internation...|NULL|
|1216527|Sharjah Cricket S...|NULL|
|1216547|Dubai Internation...|NULL|
+-------+--------------------+----+
only showing top 20 rows



### Final Null Value Handling

In [50]:
cleaned_matches = (
    bronze_matches
    # --- 1. Fix standard text nulls ---
    .withColumn("eliminator", when(col("eliminator").isNull(), lit("No")).otherwise(col("eliminator")))
    .withColumn("outcome", when(col("outcome").isNull(), lit("Normal")).otherwise(col("outcome")))
    .withColumn("method", when(col("method").isNull(), lit("Normal")).otherwise(col("method")))
    
    # --- 2. Fix numerical nulls ---
    .withColumn("winner_runs", when(col("winner_runs").isNull(), lit(0)).otherwise(col("winner_runs")))
    .withColumn("winner_wickets", when(col("winner_wickets").isNull(), lit(0)).otherwise(col("winner_wickets")))
    
    # --- 3. Add your custom 'result_type' analytical column ---
    .withColumn("result_type",
        when(col("outcome") == "tie", lit("Tie"))
        .when(col("outcome") == "no result", lit("No Result"))
        .when(col("winner_wickets") > 0, lit("Won by Wickets"))
        .when(col("winner_runs") > 0, lit("Won by Runs"))
        .otherwise(lit("Unknown"))
    )
    
    # --- 4. Handle remaining metadata/text nulls ---
    .withColumn("winner", when(col("winner").isNull(), lit("No Winner")).otherwise(col("winner")))
    .withColumn("player_of_match", when(col("player_of_match").isNull(), lit("None")).otherwise(col("player_of_match")))
    .withColumn("tv_umpire", when(col("tv_umpire").isNull(), lit("Not Recorded")).otherwise(col("tv_umpire")))
    .withColumn("reserve_umpire", when(col("reserve_umpire").isNull(), lit("Not Recorded")).otherwise(col("reserve_umpire")))
    .withColumn("match_number", when(col("match_number").isNull(), lit(0)).otherwise(col("match_number")))
    
    # Simple fix for City column if Venue contains Dubai/Sharjah
    .withColumn("city", 
        when(col("city").isNull() & col("venue").contains("Dubai"), lit("Dubai"))
        .when(col("city").isNull() & col("venue").contains("Sharjah"), lit("Sharjah"))
        .otherwise(col("city"))
    )
    
    # --- 5. Drop irrelevant columns ---
    .drop("neutralvenue", "balls_per_over", "date1", "date2", "event")
)

# Display a slice of the table to confirm everything looks good
display(cleaned_matches)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e533ee90-a6ee-4162-8023-88736c85d055)

In [51]:
# Count nulls for each column
null_counts = cleaned_matches.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in cleaned_matches.columns]
)

display(null_counts)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 98d97674-8e7c-4e53-b56e-accc203dfa2d)

### Fixing Data Types

In [52]:
cleaned_matches.printSchema()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 54, Finished, Available, Finished, False)

root
 |-- eliminator: string (nullable = true)
 |-- team1: string (nullable = true)
 |-- umpire2: string (nullable = true)
 |-- umpire1: string (nullable = true)
 |-- outcome: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- method: string (nullable = true)
 |-- date: date (nullable = true)
 |-- team2: string (nullable = true)
 |-- player_of_match: string (nullable = true)
 |-- winner_wickets: integer (nullable = true)
 |-- winner_runs: integer (nullable = true)
 |-- reserve_umpire: string (nullable = true)
 |-- season: string (nullable = true)
 |-- city: string (nullable = true)
 |-- winner: string (nullable = true)
 |-- match_number: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- match_referee: string (nullable = true)
 |-- tv_umpire: string (nullable = true)
 |-- toss_winner: string (nullable = true)
 |-- toss_decision: string (nullable = true)
 |-- matchId: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- s

In [53]:
# Looking at the Schema there is no need of the data type conversion

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 55, Finished, Available, Finished, False)

### Checking for Duplicates

In [54]:
total_rows = cleaned_matches.count()
distinct_rows = cleaned_matches.distinct().count()
duplicate_rows_count = total_rows - distinct_rows

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")
print(f"Exact Duplicate Rows: {duplicate_rows_count}")

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 56, Finished, Available, Finished, False)

Total Rows: 1095
Distinct Rows: 1095
Exact Duplicate Rows: 0


In [55]:
# Select distinct seasons and sort them
distinct_seasons = cleaned_matches.select("season").distinct().orderBy("season")

# View as a table
display(distinct_seasons)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9729dd48-de59-4a4b-831f-6903aab6149d)

In [56]:
silver_matches = (
    cleaned_matches
    # Explicitly map the messy text to the correct, clean calendar year strings
    .withColumn("season", 
        when(col("season") == "2007/08", lit("2008"))
        .when(col("season") == "2009/10", lit("2010"))
        .when(col("season") == "2020/21", lit("2020"))
        .otherwise(col("season")) # Keeps 2009, 2011, 2024, etc. exactly as they are
    )
)

# Verify the fix worked
display(cleaned_matches.select("season").distinct().orderBy("season"))

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 855551f1-d5ba-4648-a879-35323f6fe1b3)

# **Working with Deliveries Table**

In [57]:
bronze_deliveries = spark.read.table("bronze_deliveries")
display(bronze_deliveries)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45ad9b41-63db-4d29-8833-919401378b55)

In [58]:
bronze_deliveries.printSchema()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 60, Finished, Available, Finished, False)

root
 |-- matchId: integer (nullable = true)
 |-- inning: integer (nullable = true)
 |-- over_ball: double (nullable = true)
 |-- over: integer (nullable = true)
 |-- ball: integer (nullable = true)
 |-- batting_team: string (nullable = true)
 |-- bowling_team: string (nullable = true)
 |-- batsman: string (nullable = true)
 |-- non_striker: string (nullable = true)
 |-- bowler: string (nullable = true)
 |-- batsman_runs: integer (nullable = true)
 |-- extras: integer (nullable = true)
 |-- isWide: double (nullable = true)
 |-- isNoBall: double (nullable = true)
 |-- Byes: double (nullable = true)
 |-- LegByes: double (nullable = true)
 |-- Penalty: double (nullable = true)
 |-- dismissal_kind: string (nullable = true)
 |-- player_dismissed: string (nullable = true)
 |-- date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)



In [59]:
# Count null values for each column in the deliveries table
deliveries_null_counts = bronze_deliveries.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_deliveries.columns]
)

display(deliveries_null_counts)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 384afadc-e94c-4057-b08c-058e9557c137)

In [60]:
# Check distinct values for extras and dismissal columns
columns_to_check = ["isWide", "isNoBall", "Byes", "LegByes", "Penalty", "dismissal_kind"]

for col_name in columns_to_check:
    print(f"\nDistinct values in {col_name}:")
    bronze_deliveries.select(col_name).distinct().show(100, False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 62, Finished, Available, Finished, False)


Distinct values in isWide:
+------+
|isWide|
+------+
|1.0   |
|4.0   |
|NULL  |
|3.0   |
|2.0   |
|5.0   |
+------+


Distinct values in isNoBall:
+--------+
|isNoBall|
+--------+
|1.0     |
|2.0     |
|NULL    |
|5.0     |
|3.0     |
+--------+


Distinct values in Byes:
+----+
|Byes|
+----+
|1.0 |
|4.0 |
|NULL|
|3.0 |
|2.0 |
+----+


Distinct values in LegByes:
+-------+
|LegByes|
+-------+
|1.0    |
|4.0    |
|NULL   |
|3.0    |
|2.0    |
|5.0    |
+-------+


Distinct values in Penalty:
+-------+
|Penalty|
+-------+
|5.0    |
|NULL   |
+-------+


Distinct values in dismissal_kind:
+---------------------+
|dismissal_kind       |
+---------------------+
|stumped              |
|hit wicket           |
|bowled               |
|lbw                  |
|caught and bowled    |
|retired hurt         |
|caught               |
|run out              |
|retired out          |
|obstructing the field|
|NULL                 |
+---------------------+



In [61]:
# 1. Reuse the team mapping you created earlier for consistency
team_name_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Kings XI Punjab": "Punjab Kings"
}

# 2. Apply transformations to create the cleaned silver layer
cleaned_deliveries = bronze_deliveries

# Standardize team names in both columns
for old_name, new_name in team_name_mapping.items():
    cleaned_deliveries = cleaned_deliveries.replace(old_name, new_name, subset=["batting_team", "bowling_team"])

# Handle nulls, clean up types, and format names
cleaned_deliveries = (
    cleaned_deliveries
    # Fill numeric extra nulls with 0 and cast to Integer
    .withColumn("isWide", when(col("isWide").isNull(), lit(0)).otherwise(col("isWide")).cast(IntegerType()))
    .withColumn("isNoBall", when(col("isNoBall").isNull(), lit(0)).otherwise(col("isNoBall")).cast(IntegerType()))
    .withColumn("Byes", when(col("Byes").isNull(), lit(0)).otherwise(col("Byes")).cast(IntegerType()))
    .withColumn("LegByes", when(col("LegByes").isNull(), lit(0)).otherwise(col("LegByes")).cast(IntegerType()))
    .withColumn("Penalty", when(col("Penalty").isNull(), lit(0)).otherwise(col("Penalty")).cast(IntegerType()))
    
    # Handle dismissal nulls
    .withColumn("dismissal_kind", when(col("dismissal_kind").isNull(), lit("None")).otherwise(col("dismissal_kind")))
    .withColumn("player_dismissed", when(col("player_dismissed").isNull(), lit("None")).otherwise(col("player_dismissed")))
    
    # Clean string whitespaces for players to secure downstream joins/groupings
    .withColumn("batsman", trim(col("batsman")))
    .withColumn("non_striker", trim(col("non_striker")))
    .withColumn("bowler", trim(col("bowler")))
)

# Verify the changes
display(cleaned_deliveries)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 61dbaab1-7891-4609-aedc-4e0670715161)

### Checking for Duplicates

In [62]:
# 1. Total vs. Distinct rows across the entire dataset
total_rows = cleaned_deliveries.count()
distinct_rows = cleaned_deliveries.distinct().count()
exact_duplicates = total_rows - distinct_rows

print(f"Total Rows: {total_rows}")
print(f"Distinct Rows: {distinct_rows}")
print(f"Exact Duplicate Rows: {exact_duplicates}")

print("-" * 40)

# 2. Check for composite key duplicates (same ball repeated)
duplicate_balls = (
    cleaned_deliveries
    .groupBy("matchId", "inning", "over", "ball")
    .count()
    .filter(col("count") > 1)
)

print(f"Duplicate Ball Records (Composite Key): {duplicate_balls.count()}")
if duplicate_balls.count() > 0:
    duplicate_balls.show(10, False)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 64, Finished, Available, Finished, False)

Total Rows: 260920
Distinct Rows: 260917
Exact Duplicate Rows: 3
----------------------------------------
Duplicate Ball Records (Composite Key): 30
+-------+------+----+----+-----+
|matchId|inning|over|ball|count|
+-------+------+----+----+-----+
|336005 |2     |6   |1   |2    |
|419141 |2     |1   |1   |2    |
|419152 |1     |3   |1   |2    |
|1304055|2     |18  |1   |2    |
|1304083|1     |12  |1   |2    |
|1304063|1     |19  |1   |2    |
|1254117|2     |18  |1   |2    |
|1359512|2     |2   |1   |2    |
|336024 |1     |18  |1   |2    |
|1254069|2     |10  |1   |2    |
+-------+------+----+----+-----+
only showing top 10 rows



In [63]:
# 1. Drop the 3 exact duplicate rows across the dataset
final_silver_deliveries = cleaned_deliveries.distinct()

# 2. Verify the fix worked by checking the row counts again
final_rows_count = final_silver_deliveries.count()
print(f"Original Row Count: {total_rows}")
print(f"Cleaned Silver Row Count: {final_rows_count}")
print(f"Successfully Removed: {total_rows - final_rows_count} rows")

# 3. Sanity check: Ensure our valid re-bowled deliveries (the 30 counts) are safely preserved
remaining_composite_dupes = (
    final_silver_deliveries
    .groupBy("matchId", "inning", "over", "ball")
    .count()
    .filter(col("count") > 1)
    .count()
)
print(f"Valid re-bowled deliveries preserved: {remaining_composite_dupes}")

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 65, Finished, Available, Finished, False)

Original Row Count: 260920
Cleaned Silver Row Count: 260917
Successfully Removed: 3 rows
Valid re-bowled deliveries preserved: 27


###

### Working with Data Types

In [64]:
final_silver_deliveries.printSchema()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 66, Finished, Available, Finished, False)

root
 |-- matchId: integer (nullable = true)
 |-- inning: integer (nullable = true)
 |-- over_ball: double (nullable = true)
 |-- over: integer (nullable = true)
 |-- ball: integer (nullable = true)
 |-- batting_team: string (nullable = true)
 |-- bowling_team: string (nullable = true)
 |-- batsman: string (nullable = true)
 |-- non_striker: string (nullable = true)
 |-- bowler: string (nullable = true)
 |-- batsman_runs: integer (nullable = true)
 |-- extras: integer (nullable = true)
 |-- isWide: integer (nullable = true)
 |-- isNoBall: integer (nullable = true)
 |-- Byes: integer (nullable = true)
 |-- LegByes: integer (nullable = true)
 |-- Penalty: integer (nullable = true)
 |-- dismissal_kind: string (nullable = true)
 |-- player_dismissed: string (nullable = true)
 |-- date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file_name: string (nullable = true)



In [65]:
# Everything is fine

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 67, Finished, Available, Finished, False)

### Quality Checks

In [66]:
# Check for orphan deliveries
orphan_deliveries = final_silver_deliveries.join(
    cleaned_matches, 
    on="matchId", 
    how="left_anti"
)

orphan_count = orphan_deliveries.count()
print(f" Quality Check 1: Orphan deliveries found: {orphan_count}")
assert orphan_count == 0, "Validation Failed: Deliveries exist without a corresponding match record!"

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 68, Finished, Available, Finished, False)

 Quality Check 1: Orphan deliveries found: 0


In [67]:
# Validate over boundaries (0 to 19)
invalid_overs = final_silver_deliveries.filter((col("over") < 0) | (col("over") > 19)).count()
invalid_balls = final_silver_deliveries.filter((col("ball") < 1) | (col("ball") > 12)).count() # 12 allows for extreme wide/no-ball over extensions

print(f" Quality Check 2: Invalid over numbers: {invalid_overs}")
print(f" Quality Check 2b: Invalid ball numbers: {invalid_balls}")

assert invalid_overs == 0, "Validation Failed: Out-of-bounds over numbers detected!"

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 69, Finished, Available, Finished, False)

 Quality Check 2: Invalid over numbers: 0
 Quality Check 2b: Invalid ball numbers: 0


In [68]:
# Total validation: batsman_runs + extras should equal the expected total per ball if you pre-calculate it, 
# or ensure individual runs are non-negative.
negative_runs = final_silver_deliveries.filter(
    (col("batsman_runs") < 0) | 
    (col("extras") < 0)
).count()

print(f" Quality Check 3: Negative run values found: {negative_runs}")
assert negative_runs == 0, "Validation Failed: Runs or extras cannot be negative numbers!"

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 70, Finished, Available, Finished, False)

 Quality Check 3: Negative run values found: 0


In [69]:
# Get distinct teams from both tables to verify alignment
match_teams = set(
    [row['team1'] for row in cleaned_matches.select("team1").distinct().collect()] +
    [row['team2'] for row in cleaned_matches.select("team2").distinct().collect()]
)

delivery_teams = set(
    [row['batting_team'] for row in final_silver_deliveries.select("batting_team").distinct().collect()] +
    [row['bowling_team'] for row in final_silver_deliveries.select("bowling_team").distinct().collect()]
)

unaligned_teams = delivery_teams - match_teams
print(f" Quality Check 4: Unaligned team names: {unaligned_teams}")
assert len(unaligned_teams) == 0, f"Validation Failed: Teams in deliveries {unaligned_teams} do not match the matches table!"

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 71, Finished, Available, Finished, False)

 Quality Check 4: Unaligned team names: set()


In [76]:
season_df = silver_matches.select("matchId", "season")

silver_deliveries = final_silver_deliveries.join(
    season_df,
    on="matchId",
    how="left"
)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 78, Finished, Available, Finished, False)

In [77]:
display(silver_deliveries)

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0306b94c-527b-4649-a8b5-bd369938537e)

In [78]:
# Write the delivery data to the Silver layer, partitioned by season
(
    silver_deliveries.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("season")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_deliveries")
)

print("silver_deliveries table successfully saved and partitioned by season!")

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 80, Finished, Available, Finished, False)

silver_deliveries table successfully saved and partitioned by season!


In [79]:
# Write the delivery data to the Silver layer, partitioned by season
(
    silver_matches.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("season")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_matches")
)

print("silver_matches table successfully saved and partitioned by season!")

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 81, Finished, Available, Finished, False)

silver_matches table successfully saved and partitioned by season!


In [81]:
spark.sql("SHOW PARTITIONS silver_deliveries").show()

StatementMeta(, 36b8be95-c309-4c1c-a611-a8c33e840bde, 83, Finished, Available, Finished, False)

+-----------+
|  partition|
+-----------+
|season=2008|
|season=2009|
|season=2010|
|season=2011|
|season=2012|
|season=2013|
|season=2014|
|season=2015|
|season=2016|
|season=2017|
|season=2018|
|season=2019|
|season=2020|
|season=2021|
|season=2022|
|season=2023|
|season=2024|
+-----------+

